# 第 8 节：SARSA 与 Q-Learning — 同策略 vs 异策略 TD 控制

---

## 📍 本节位置

```
TD Learning (07) → **SARSA / Q-Learning (08)** → Value Function Approx (09) → ...
                        ↑
                   你在这里
```

**核心主线**：上一节我们学习了 TD 预测（给定策略下估计价值函数），
现在扩展到 **TD 控制**——既估计价值函数，又改进策略。

本节将对比四种 TD 控制算法：
| 算法 | 类型 | 核心特点 |
|------|------|----------|
| **SARSA** | 同策略 (On-policy) | 学习行为策略的价值 |
| **Q-Learning** | 异策略 (Off-policy) | 直接学习最优策略 |
| **Expected SARSA** | 同策略的变体 | 降低方差，介于两者之间 |
| **Double Q-Learning** | 异策略的改进 | 解决最大化偏差 |


## 🎯 学习目标

1. **理解同策略 vs 异策略**：SARSA（同策略）与 Q-Learning（异策略）的本质区别
2. **掌握 SARSA 算法**：更新公式、伪代码、在 CliffWalking 上的行为
3. **掌握 Q-Learning 算法**：更新公式、伪代码、异策略学习的优势
4. **理解 CliffWalking 实验结果**：SARSA 走安全路径，Q-Learning 走最优（但危险）路径
5. **掌握 Expected SARSA**：如何降低 SARSA 的方差
6. **理解最大化偏差 (Maximization Bias)**：为什么 max 会导致过估计
7. **掌握 Double Q-Learning**：如何用双估计解决最大化偏差
8. **能对比四种算法的适用场景**


## 1. 同策略 vs 异策略：回顾

### 核心区别

| 概念 | 同策略 (On-policy) | 异策略 (Off-policy) |
|------|-------------------|-------------------|
| **要评估/改进的策略** | 当前正在执行的策略 | 目标策略（通常是最优策略） |
| **行为策略** | 与目标策略相同 | 与目标策略不同（通常含更多探索） |
| **数据来源** | 只能使用当前策略生成的数据 | 可以使用任意策略生成的数据 |
| **样本效率** | 较低（旧数据不能用） | 较高（可复用历史数据） |
| **稳定性** | 更稳定（目标即行为） | 需处理分布偏移 |

### 数学表达式

**同策略** 评估的是行为策略 $\pi$ 的价值函数 $V^\pi$：
$$Q^\pi(s, a) = \mathbb{E}_\pi[R_{t+1} + \gamma Q^\pi(S_{t+1}, A_{t+1}) \mid S_t=s, A_t=a]$$

**异策略** 评估的是目标策略 $\pi^*$ 的价值函数 $Q^*$，即使数据由不同策略 $\mu$ 生成：
$$Q^*(s, a) = \mathbb{E}_\mu[R_{t+1} + \gamma \max_{a'} Q^*(S_{t+1}, a') \mid S_t=s, A_t=a]$$

### SARSA vs Q-Learning 一句话概括

- **SARSA**：「我按当前策略走一步，然后用下一步的实际动作来更新」
- **Q-Learning**：「我按当前策略走一步，但更新时假设下一步是最优的」


## 2. SARSA 算法

### 名称由来

**S**tate-**A**ction-**R**eward-**S**tate-**A**ction

### 更新公式

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma Q(S', A') - Q(S, A) \right]$$

其中：
- $(S, A)$ 是当前状态-动作对
- $R$ 是执行 $A$ 后得到的奖励
- $(S', A')$ 是 **实际执行** 的下一步状态-动作对
- $\alpha$ 是学习率，$\gamma$ 是折扣因子

### 伪代码

```
Initialize Q(s, a) arbitrarily for all s ∈ S, a ∈ A(s)

Loop for each episode:
    Initialize S
    Choose A from S using ε-greedy policy derived from Q
    Loop for each step of episode:
        Take action A, observe R, S'
        Choose A' from S' using ε-greedy policy derived from Q
        Q(S, A) ← Q(S, A) + α [R + γ Q(S', A') - Q(S, A)]
        S ← S', A ← A'
    until S is terminal
```

### 关键特性

- **完全使用行为策略产生的数据**：$(S, A, R, S', A')$ 全部来自同一个策略
- **同策略**：评估和改进的是同一个策略（ε-greedy）
- **收敛条件**：当所有 $(s, a)$ 被无限次访问且策略在极限处收敛到贪婪策略时


## 3. Q-Learning 算法

### 核心创新：异策略 TD 控制

Watkins (1989) 提出了革命性的想法：**更新时不需要使用实际采取的 $A'$**。

### 更新公式

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \max_{a} Q(S', a) - Q(S, A) \right]$$

其中关键区别在于 $\max_a Q(S', a)$ 而不是 $Q(S', A')$。

### 伪代码

```
Initialize Q(s, a) arbitrarily for all s ∈ S, a ∈ A(s)

Loop for each episode:
    Initialize S
    Loop for each step of episode:
        Choose A from S using ε-greedy policy derived from Q
        Take action A, observe R, S'
        Q(S, A) ← Q(S, A) + α [R + γ max_a Q(S', a) - Q(S, A)]
        S ← S'
    until S is terminal
```

### 为什么 Q-Learning 是异策略？

- **行为策略**：ε-greedy（用于生成数据）
- **目标策略**：贪婪策略 $\pi^*(s) = \arg\max_a Q(s, a)$（用于计算目标）
- 更新公式中的 $\max_a Q(S', a)$ 相当于假设下一步采用贪婪策略——与行为策略无关

### 重要定理

**Q-Learning 的最优性保证**：只要每个 $(s, a)$ 被无限次访问，且学习率满足 Robbins-Monro 条件
（$\sum \alpha_t = \infty$，$\sum \alpha_t^2 < \infty$），
Q-Learning 以概率 1 收敛到 $Q^*$。


In [ ]:
# ⚙️ 导入和环境设置
import sys
sys.path.insert(0, "/workspace/data/vggt-omega/rl")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import gymnasium as gym
from tqdm import tqdm

from rl_course.utils.seeding import set_seed
from rl_course.agents.tabular import (
    TabularSARSAAgent,
    TabularQLearningAgent,
    TabularExpectedSARSAAgent,
    TabularDoubleQLearningAgent,
)

set_seed(42)

# 确保输出目录存在
import os
os.makedirs("outputs/figures", exist_ok=True)

print("✅ 导入成功")


### CliffWalking 环境

经典的 **Cliff Walking** 环境来自 Sutton & Barto (2018) Example 6.6：

```
                    路径
                    ↑
  ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
  │  │  │  │  │  │  │  │  │  │  │  │ G│  ← 目标
  ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
  │  │  │  │  │  │  │  │  │  │  │  │  │
  ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
  │S │💀│💀│💀│💀│💀│💀│💀│💀│💀│💀│  │
  └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘
   ↑
  起点       悬崖（reward = -100）
```

- **网格**：4 行 × 12 列（48 个状态）
- **动作**：上 (0)、右 (1)、下 (2)、左 (3)
- **起点**：左下角 (3, 0)
- **目标**：右下角 (3, 11)
- **悬崖**：底部从 (3, 1) 到 (3, 10)
- **每步奖励**：-1
- **掉入悬崖**：-100 并回到起点
- **Episode 终止**：到达目标或掉入悬崖

我们将看到：
- **SARSA** 学会沿悬崖上方的安全路径走（保守但安全）
- **Q-Learning** 学会沿悬崖边缘的最优路径走（捷径但冒险）


In [ ]:
# ===== 自定义 CliffWalking 环境 =====
# 我们自行实现以更好地控制行为和可视化

class CliffWalkingEnv:
    """
    Cliff Walking 环境 (Sutton & Barto, Example 6.6)

    网格: 4 × 12
      行 0: 安全走廊
      行 1-2: 安全区域
      行 3: 起点(左) → 悬崖(中间) → 目标(右)

    动作: 0=上, 1=右, 2=下, 3=左
    """

    def __init__(self, nrows=4, ncols=12):
        self.nrows = nrows
        self.ncols = ncols
        self.n_states = nrows * ncols
        self.n_actions = 4

        # 起点 (3, 0)，目标 (3, 11)
        self.start_state = 3 * ncols + 0      # 索引 36
        self.goal_state = 3 * ncols + 11      # 索引 47

        # 悬崖区域: 行 3, 列 1..10
        self.cliff_states = set()
        for c in range(1, 11):
            self.cliff_states.add(3 * ncols + c)

        self.current_state = self.start_state

    def _to_rc(self, s):
        return s // self.ncols, s % self.ncols

    def _to_state(self, r, c):
        return r * self.ncols + c

    def reset(self):
        self.current_state = self.start_state
        return self.current_state, {}

    def step(self, action):
        """执行动作，返回 (next_state, reward, terminated, truncated, info)"""
        r, c = self._to_rc(self.current_state)

        # 动作效果
        if action == 0:    # 上
            r = max(0, r - 1)
        elif action == 1:  # 右
            c = min(self.ncols - 1, c + 1)
        elif action == 2:  # 下
            r = min(self.nrows - 1, r + 1)
        elif action == 3:  # 左
            c = max(0, c - 1)

        next_state = self._to_state(r, c)

        # 检查是否掉入悬崖
        if next_state in self.cliff_states:
            reward = -100.0
            terminated = False  # Cliff does NOT terminate episode!
            next_state = self.start_state  # 回到起点
        elif next_state == self.goal_state:
            reward = -1.0  # Same as any step
            terminated = True
        else:
            reward = -1.0
            terminated = False

        self.current_state = next_state
        return next_state, reward, terminated, False, {"pos": (r, c)}

    def render_grid(self, agent_path=None):
        """以文本形式渲染网格"""
        grid = [['·' for _ in range(self.ncols)] for _ in range(self.nrows)]

        # 目标
        gr, gc = self._to_rc(self.goal_state)
        grid[gr][gc] = 'G'

        # 起点
        sr, sc = self._to_rc(self.start_state)
        grid[sr][sc] = 'S'

        # 悬崖
        for cs in self.cliff_states:
            r, c = self._to_rc(cs)
            grid[r][c] = 'X'

        # 路径
        if agent_path:
            for s in agent_path:
                r, c = self._to_rc(s)
                if grid[r][c] in ('·',):
                    grid[r][c] = '●'
                elif grid[r][c] == '●':
                    pass

        lines = []
        for r in range(self.nrows):
            lines.append(' '.join(grid[r]))
        return '\n'.join(lines)


# ===== 训练函数 =====
def train_agent_on_cliff(agent, env, n_episodes=500, max_steps=200):
    """训练智能体并记录每 episode 的累积奖励"""
    returns = []
    episode_lengths = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False
        total_reward = 0.0
        steps = 0

        if isinstance(agent, TabularSARSAAgent):
            # SARSA 需要额外选择第一个动作
            action = agent.act(state, train=True)

        while not (terminated or truncated) and steps < max_steps:
            if isinstance(agent, TabularSARSAAgent):
                # SARSA 用 (s, a, r, s', a') 更新
                next_state, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                steps += 1

                if terminated or truncated:
                    agent.update(state, action, reward, next_state, 0, done=(terminated or truncated), terminated=terminated)
                else:
                    next_action = agent.act(next_state, train=True)
                    agent.update(state, action, reward, next_state, next_action, done=False, terminated=False)
                    state = next_state
                    action = next_action
            else:
                # Q-Learning / Expected SARSA / Double Q-Learning
                action = agent.act(state, train=True)
                next_state, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                steps += 1
                agent.update(state, action, reward, next_state, done=(terminated or truncated), terminated=terminated)
                state = next_state

        returns.append(total_reward)
        episode_lengths.append(steps)

    return returns, episode_lengths


def extract_optimal_path(agent, env):
    """提取贪婪策略下的最优路径"""
    state, _ = env.reset()
    path = [state]
    visited = set()

    for _ in range(200):
        if state in visited:
            break
        visited.add(state)
        action = np.argmax(agent.Q[state])
        next_state, _, terminated, truncated, _ = env.step(action)
        path.append(next_state)
        if terminated or truncated:
            break
        state = next_state
    return path


def get_policy_map(agent, env):
    """获取每个状态的最优动作"""
    policy = np.zeros(env.n_states, dtype=np.int32)
    for s in range(env.n_states):
        policy[s] = np.argmax(agent.Q[s])
    return policy


# 测试环境
test_env = CliffWalkingEnv()
s, _ = test_env.reset()
print("初始状态:", s)
print("环境布局:")
print(test_env.render_grid())
print(f"\n动作空间大小: {test_env.n_actions}")
print(f"状态空间大小: {test_env.n_states}")
print(f"悬崖状态数: {len(test_env.cliff_states)}")


## 4. SARSA 在 CliffWalking 上的表现

SARSA 是同策略算法，它**亲身体验**掉入悬崖的痛苦，因此学会**绕开悬崖走安全路径**。


In [ ]:
# ===== 训练 SARSA =====
set_seed(42)
env_sarsa = CliffWalkingEnv()
sarsa_agent = TabularSARSAAgent(
    n_states=env_sarsa.n_states,
    n_actions=env_sarsa.n_actions,
    gamma=0.99,
    alpha=0.5,
    epsilon=0.1,
    seed=42,
)

print("训练 SARSA...")
sarsa_returns, sarsa_lengths = train_agent_on_cliff(
    sarsa_agent, env_sarsa, n_episodes=500
)
print(f"SARSA 训练完成！")
print(f"最后 50 episode 平均回报: {np.mean(sarsa_returns[-50:]):.1f}")
print(f"最后 50 episode 平均步数: {np.mean(sarsa_lengths[-50:]):.1f}")

# 提取 SARSA 的最优路径
sarsa_path = extract_optimal_path(sarsa_agent, env_sarsa)
print(f"\nSARSA 最优路径长度: {len(sarsa_path)} 步")
print("路径（S=起点, X=悬崖, G=目标, ●=路径）：")
print(env_sarsa.render_grid(sarsa_path))


## 5. Q-Learning 在 CliffWalking 上的表现

Q-Learning 是异策略算法，它假设**下一步采取最优动作**来更新当前 Q 值。
因此它学习的是**最优策略**——沿着悬崖边缘走最短路径——尽管这条路径风险更高。


In [ ]:
# ===== 训练 Q-Learning =====
set_seed(42)
env_ql = CliffWalkingEnv()
ql_agent = TabularQLearningAgent(
    n_states=env_ql.n_states,
    n_actions=env_ql.n_actions,
    gamma=0.99,
    alpha=0.5,
    epsilon=0.1,
    seed=42,
)

print("训练 Q-Learning...")
ql_returns, ql_lengths = train_agent_on_cliff(
    ql_agent, env_ql, n_episodes=500
)
print(f"Q-Learning 训练完成！")
print(f"最后 50 episode 平均回报: {np.mean(ql_returns[-50:]):.1f}")
print(f"最后 50 episode 平均步数: {np.mean(ql_lengths[-50:]):.1f}")

# 提取 Q-Learning 的最优路径
ql_path = extract_optimal_path(ql_agent, env_ql)
print(f"\nQ-Learning 最优路径长度: {len(ql_path)} 步")
print("路径（S=起点, X=悬崖, G=目标, ●=路径）：")
print(env_ql.render_grid(ql_path))


## 6. SARSA vs Q-Learning 对比分析

### 关键发现

| 指标 | SARSA | Q-Learning |
|------|-------|-----------|
| 学习路径 | 沿悬崖上方安全绕行 | 沿悬崖边缘冒险走 |
| 最优路径长度 | 较长（约 13 步） | 较短（约 12 步） |
| 训练中掉入悬崖次数 | 少（保守策略） | 多（仍在试探边缘） |
| 收敛后是否靠近悬崖 | 否 | 是 |
| 泛化风险 | 低 | 高（ε 探索时可能掉下悬崖） |

### 为什么 SARSA 更保守？

因为 SARSA 在更新时使用 $Q(S', A')$，其中 $A'$ 是用 ε-greedy 选择的。
如果行为策略在悬崖边缘选择「向下」掉入悬崖的动作，SARSA 会将该状态的 Q 值拉低，
从而学会**彻底远离悬崖**。

而 Q-Learning 在更新时使用 $\max_a Q(S', a)$，它总是假设下一步选择**最优**动作。
即使行为策略有 ε 概率掉下悬崖，更新目标仍然是「不掉下去的最优动作」。
因此 Q-Learning 更敢于靠近悬崖边缘。


In [ ]:
# ===== 对比学习曲线 =====
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 回报曲线
ax = axes[0]
window = 20
eps_range = range(len(sarsa_returns))

# 原始曲线
ax.plot(eps_range, sarsa_returns, alpha=0.1, color="steelblue")
ax.plot(eps_range, ql_returns, alpha=0.1, color="coral")

# 滑动平均
sarsa_smoothed = np.convolve(sarsa_returns, np.ones(window)/window, mode="valid")
ql_smoothed = np.convolve(ql_returns, np.ones(window)/window, mode="valid")
ax.plot(range(window-1, len(sarsa_returns)), sarsa_smoothed,
        color="steelblue", linewidth=2, label="SARSA")
ax.plot(range(window-1, len(ql_returns)), ql_smoothed,
        color="coral", linewidth=2, label="Q-Learning")

ax.set_xlabel("Episode"); ax.set_ylabel("Return")
ax.set_title("Learning Curves (SARSA vs Q-Learning)")
ax.legend(); ax.grid(True, alpha=0.3)

# 2. 累积回报直方图
ax = axes[1]
final_sarsa = sarsa_returns[-100:]
final_ql = ql_returns[-100:]
ax.hist(final_sarsa, bins=15, alpha=0.6, color="steelblue", label=f"SARSA (mean={np.mean(final_sarsa):.1f})")
ax.hist(final_ql, bins=15, alpha=0.6, color="coral", label=f"Q-Learning (mean={np.mean(final_ql):.1f})")
ax.set_xlabel("Return"); ax.set_ylabel("Frequency")
ax.set_title("Final 100 Episodes Return Distribution")
ax.legend(); ax.grid(True, alpha=0.3)

# 3. 收敛过程中的最低回报（掉悬崖情况）
ax = axes[2]
min_window = 50
sarsa_mins = [np.min(sarsa_returns[max(0, i-min_window):i+1]) for i in range(len(sarsa_returns))]
ql_mins = [np.min(ql_returns[max(0, i-min_window):i+1]) for i in range(len(ql_returns))]
ax.plot(sarsa_mins, color="steelblue", linewidth=1.5, label="SARSA min (50-ep window)")
ax.plot(ql_mins, color="coral", linewidth=1.5, label="Q-Learning min (50-ep window)")
ax.axhline(-100, color="gray", linestyle="--", alpha=0.5, label="Cliff penalty")
ax.set_xlabel("Episode"); ax.set_ylabel("Min Return (50-ep window)")
ax.set_title("Frequency of Falling off Cliff")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/figures/08_sarsa_vs_ql_curves.png")
plt.close()
print("✅ SARSA vs Q-Learning 对比图已保存")


In [ ]:
# ===== 策略可视化 =====
def plot_policy_comparison(sarsa_agent, ql_agent, env, filepath):
    """对比 SARSA 和 Q-Learning 学到的策略"""
    fig, axes = plt.subplots(2, 1, figsize=(14, 9))

    action_symbols = ['↑', '→', '↓', '←']
    agents = [('SARSA', sarsa_agent), ('Q-Learning', ql_agent)]
    colors = ['steelblue', 'coral']

    for idx, (name, agent) in enumerate(agents):
        ax = axes[idx]
        policy = np.zeros(env.n_states, dtype=np.int32)
        for s in range(env.n_states):
            policy[s] = np.argmax(agent.Q[s])

        grid = policy.reshape(env.nrows, env.ncols)

        # 创建颜色网格背景（基于 Q 值最大差异）
        q_max = np.max(agent.Q, axis=1).reshape(env.nrows, env.ncols)
        q_min = np.min(agent.Q, axis=1).reshape(env.nrows, env.ncols)
        q_range = q_max - q_min
        q_range = (q_range - q_range.min()) / (q_range.max() - q_range.min() + 1e-8)

        ax.imshow(q_range, cmap='Blues', alpha=0.3, aspect='equal')

        # 给格子划线
        for r in range(env.nrows):
            for c in range(env.ncols):
                s = env._to_state(r, c)
                a = grid[r, c]

                # 悬崖用特殊标记
                if s in env.cliff_states:
                    ax.add_patch(Rectangle((c-0.5, r-0.5), 1, 1,
                                           facecolor='red', alpha=0.5))
                    ax.text(c, r, '💀', ha='center', va='center', fontsize=14)
                    continue

                if s == env.goal_state:
                    ax.add_patch(Rectangle((c-0.5, r-0.5), 1, 1,
                                           facecolor='gold', alpha=0.5))
                    ax.text(c, r, 'G', ha='center', va='center', fontsize=14, fontweight='bold')
                    continue

                if s == env.start_state:
                    ax.add_patch(Rectangle((c-0.5, r-0.5), 1, 1,
                                           facecolor='lightgreen', alpha=0.5))

                # 画动作箭头
                ax.text(c, r, action_symbols[a], ha='center', va='center',
                        fontsize=16, color=colors[idx], fontweight='bold')

        ax.set_xlim(-0.5, env.ncols - 0.5)
        ax.set_ylim(env.nrows - 0.5, -0.5)
        ax.set_xticks(range(env.ncols))
        ax.set_yticks(range(env.nrows))
        ax.set_title(f"{name} — Learned Policy")
        ax.set_xlabel("Column")
        ax.set_ylabel("Row")
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()
    print(f"✅ 策略对比图已保存: {filepath}")

plot_policy_comparison(sarsa_agent, ql_agent, env_sarsa,
                       "outputs/figures/08_policy_comparison.png")


## 7. Expected SARSA

### 动机

SARSA 的更新使用 $Q(S', A')$，即**一个采样**的动作价值。这带来了方差：
- 如果 $A'$ 恰好选到较差的动作，更新会过度悲观
- 如果 $A'$ 恰好选到较好的动作，更新会过度乐观

### Expected SARSA 的核心想法

用 **期望价值** 代替 **采样价值**：

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \mathbb{E}_\pi[Q(S', \cdot)] - Q(S, A) \right]$$

其中：
$$\mathbb{E}_\pi[Q(S', \cdot)] = \sum_{a'} \pi(a'|S') \, Q(S', a')$$

对于 ε-greedy 策略：
$$\mathbb{E}_\pi[Q(S', \cdot)] = (1-\varepsilon) \cdot \max_a Q(S', a) + \frac{\varepsilon}{|\mathcal{A}|} \sum_a Q(S', a)$$

### 与 SARSA 和 Q-Learning 的关系

| 算法 | 使用的 Target | 方差 | 偏差 |
|------|-------------|------|------|
| **SARSA** | $R + \gamma Q(S', A')$ | 较高（单样本） | 低 |
| **Expected SARSA** | $R + \gamma \mathbb{E}_\pi[Q(S', \cdot)]$ | 低（期望值） | 低 |
| **Q-Learning** | $R + \gamma \max_a Q(S', a)$ | 中等 | 高（最大化偏差） |

Expected SARSA 通常介于 SARSA 和 Q-Learning 之间：
- 当 $\varepsilon \to 0$ 时，Expected SARSA → Q-Learning（因为策略接近贪婪）
- 当 $\varepsilon \to 1$ 时，Expected SARSA 完全随机，价值趋向平均值

### Expected SARSA 的优势

1. **方差更低**：使用期望而非采样，消除了 $A'$ 的随机性
2. **可以在不改变学习率的情况下使用更大的 $\varepsilon$**
3. **是 SARSA 和 Q-Learning 之间的平滑桥梁**


In [ ]:
# ===== Expected SARSA 在 CliffWalking 上的表现 =====
set_seed(42)
env_esarsa = CliffWalkingEnv()
esarsa_agent = TabularExpectedSARSAAgent(
    n_states=env_esarsa.n_states,
    n_actions=env_esarsa.n_actions,
    gamma=0.99,
    alpha=0.5,
    epsilon=0.1,
    seed=42,
)

print("训练 Expected SARSA...")
esarsa_returns, esarsa_lengths = train_agent_on_cliff(
    esarsa_agent, env_esarsa, n_episodes=500
)
print(f"Expected SARSA 训练完成！")
print(f"最后 50 episode 平均回报: {np.mean(esarsa_returns[-50:]):.1f}")

# 对比三种算法
window = 20
fig, ax = plt.subplots(figsize=(10, 5))

for name, returns, color in [
    ("SARSA", sarsa_returns, "steelblue"),
    ("Expected SARSA", esarsa_returns, "seagreen"),
    ("Q-Learning", ql_returns, "coral"),
]:
    smoothed = np.convolve(returns, np.ones(window)/window, mode="valid")
    ax.plot(range(window-1, len(returns)), smoothed,
            color=color, linewidth=2, label=name)

ax.set_xlabel("Episode"); ax.set_ylabel("Return (smoothed)")
ax.set_title("SARSA vs Expected SARSA vs Q-Learning")
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig("outputs/figures/08_expected_sarsa_comparison.png")
plt.close()
print("✅ Expected SARSA 对比图已保存")


## 8. 最大化偏差 (Maximization Bias)

### 问题描述

Q-Learning 使用 $\max_a Q(S', a)$ 作为 target。问题在于：

$$\mathbb{E}[\max_a Q(S', a)] \geq \max_a \mathbb{E}[Q(S', a)]$$

当 $Q(S', a)$ 是**有噪声的估计**时，取最大值倾向于**选择被高估的动作**。

### 直观例子

假设在状态 $S'$ 中有两个动作 $a_1, a_2$，真实价值都是 0：
- $Q(S', a_1) = 0 + \text{噪声}_1$
- $Q(S', a_2) = 0 + \text{噪声}_2$

即使 $\mathbb{E}[\text{噪声}] = 0$，但：
$$\mathbb{E}[\max(Q(S', a_1), Q(S', a_2))] > 0$$

因为 max 会选到噪声为正的那个！

### 后果

- Q-Learning 系统性高估动作价值
- 在随机环境中，这种高估可能导致次优策略
- 错误高估的动作更难被纠正（因为高估导致被选中，被选中又进一步强化了高估）

### Thrun & Schwartz (1993) 的发现

最大化偏差的程度与**动作数量**和**估计方差**成正比：
$$\text{bias} \propto \sigma \cdot \sqrt{\log |\mathcal{A}|}$$

动作越多，偏差越大。


In [ ]:
# ===== 演示最大化偏差 =====
# 简单环境：一个起始状态，两个动作，真实 Q 值都是 0
# 观察 Q-Learning 的系统性高估

set_seed(42)
np.random.seed(42)

def run_maximization_bias_experiment(n_actions=10, n_steps=1000, n_runs=500):
    """
    演示最大化偏差。

    假设所有动作的真实 Q 值 = 0，奖励服从 N(0, 1)。
    观察 Q-Learning 的平均估计值。
    """
    n_states = 2  # 起始 + 结束
    alpha = 0.1
    gamma = 0.99

    # Q-Learning 的 Q 表
    q_values_over_time = np.zeros((n_runs, n_steps))

    for run in range(n_runs):
        Q = np.zeros((n_states, n_actions))

        for step in range(n_steps):
            state = 0
            action = np.random.randint(n_actions)  # 随机选择

            # 奖励来自 N(0, 1)，但真实期望是 0
            reward = np.random.randn()

            # 进入终止状态
            next_state = 1  # 终止
            terminated = True

            # Q-Learning 更新
            target = reward + gamma * (1 - float(terminated)) * np.max(Q[next_state])
            Q[state, action] += alpha * (target - Q[state, action])

        # 记录对第一个动作的估计值
        q_values_over_time[run, :] = Q[0, 0]  # 任意选一个动作跟踪

    return q_values_over_time


# 实验：不同动作数量
n_actions_list = [2, 5, 10, 25]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, n_actions in enumerate(n_actions_list):
    ax = axes[idx // 2, idx % 2]
    q_vals = run_maximization_bias_experiment(n_actions=n_actions, n_steps=200, n_runs=100)

    # 计算均值和置信区间
    mean_q = np.mean(q_vals, axis=0)
    std_q = np.std(q_vals, axis=0)

    ax.plot(mean_q, linewidth=2, color="coral")
    ax.fill_between(range(len(mean_q)),
                    mean_q - 1.96 * std_q / np.sqrt(100),
                    mean_q + 1.96 * std_q / np.sqrt(100),
                    alpha=0.2, color="coral")
    ax.axhline(0, color="gray", linestyle="--", linewidth=1, label="True Q=0")
    ax.set_xlabel("Step"); ax.set_ylabel("Estimated Q(0, a₀)")
    ax.set_title(f"{n_actions} actions — Maximization Bias")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/figures/08_maximization_bias.png")
plt.close()
print("✅ 最大化偏差演示图已保存")
print("")
print("观察：动作数量越多，Q-Learning 的高估越严重！")
print("这是因为 max 在更多样本中总是选出被正噪声污染的那个。")


## 9. Double Q-Learning

### 核心思想：解耦选择与评估

最大化偏差的根源是**同一个 Q 表既用于选择最优动作，又用于评估该动作的价值**。

Double Q-Learning (Hasselt, 2010) 的解决方案是**用两个独立的 Q 表**：

| 角色 | Q_A | Q_B |
|------|-----|-----|
| 选择 (Selection) | ✓ | ✓ |
| 评估 (Evaluation) | ✓ (用 Q_B 评价 Q_A 选出的动作) | ✓ (用 Q_A 评价 Q_B 选出的动作) |

### 更新公式

以更新 $Q_A$ 为例：
1. **选择**：$a^* = \arg\max_a Q_A(S', a)$ — 用 $Q_A$ 选择最优动作
2. **评估**：用 $Q_B$ 评估：target $= R + \gamma Q_B(S', a^*)$

$$Q_A(S, A) \leftarrow Q_A(S, A) + \alpha \left[ R + \gamma Q_B(S', \arg\max_a Q_A(S', a)) - Q_A(S, A) \right]$$

对称地更新 $Q_B$：
$$Q_B(S, A) \leftarrow Q_B(S, A) + \alpha \left[ R + \gamma Q_A(S', \arg\max_a Q_B(S', a)) - Q_B(S, A) \right]$$

每次更新**随机选择**更新 $Q_A$ 或 $Q_B$。

### 为什么有效？

当 $Q_A$ 高估了某个动作时，$Q_B$ 对其的估计是独立的（可能没有高估）。
因此 $\mathbb{E}[Q_B(S', \arg\max_a Q_A(S', a))] \leq \max_a \mathbb{E}[Q(S', a)]$，
从而减少了正偏差。

### Double Q-Learning 与 Double DQN

Double Q-Learning 的思想直接启发了深度 RL 中的 **Double DQN**：
- 当前网络 $\theta$ 选择动作
- 目标网络 $\theta^-$ 评估动作


In [ ]:
# ===== Double Q-Learning 对比实验 =====
set_seed(42)
env_dql = CliffWalkingEnv()
dql_agent = TabularDoubleQLearningAgent(
    n_states=env_dql.n_states,
    n_actions=env_dql.n_actions,
    gamma=0.99,
    alpha=0.5,
    epsilon=0.1,
    seed=42,
)

print("训练 Double Q-Learning...")
dql_returns, dql_lengths = train_agent_on_cliff(
    dql_agent, env_dql, n_episodes=500
)
print(f"Double Q-Learning 训练完成！")
print(f"最后 50 episode 平均回报: {np.mean(dql_returns[-50:]):.1f}")

# 对比 Q-Learning 和 Double Q-Learning 的 Q 值估计
# 在训练结束后，比较两种算法对最优动作的 Q 值估计
print("\n=== Q 值估计对比 ===")
true_q = -13.0  # CliffWalking 最优路径的近似真实值

# 从起点评估
s = env_dql.start_state
ql_best_q = np.max(ql_agent.Q[s])
dql_best_q = np.max(dql_agent.Q[s])
dql_qa_best = np.max(dql_agent.QA[s])
dql_qb_best = np.max(dql_agent.QB[s])

print(f"Q-Learning 对起点的 max Q 估计: {ql_best_q:.2f}")
print(f"Double Q-Learning Q_A 对起点的 max Q 估计: {dql_qa_best:.2f}")
print(f"Double Q-Learning Q_B 对起点的 max Q 估计: {dql_qb_best:.2f}")
print(f"Double Q-Learning 平均 Q 估计: {dql_best_q:.2f}")
print(f"预期真实 Q 值参考: ≈ {true_q:.2f}")


## 10. 四种算法综合对比

### 实验设置

在 CliffWalking 上对比四种算法的完整表现：
- **SARSA** (同策略 TD)
- **Q-Learning** (异策略 TD)
- **Expected SARSA** (同策略，使用期望)
- **Double Q-Learning** (改进版异策略)

我们将观察：
1. 学习曲线收敛速度
2. 最终策略的优劣
3. 对学习率 α 的敏感性
4. 对探索率 ε 的敏感性


In [ ]:
# ===== 四种算法综合比较 =====
set_seed(42)
n_episodes = 500
n_runs = 5  # 多次运行取平均

algorithms = {
    "SARSA": TabularSARSAAgent,
    "Q-Learning": TabularQLearningAgent,
    "Expected SARSA": TabularExpectedSARSAAgent,
    "Double Q-Learning": TabularDoubleQLearningAgent,
}

colors = {
    "SARSA": "steelblue",
    "Q-Learning": "coral",
    "Expected SARSA": "seagreen",
    "Double Q-Learning": "purple",
}

all_returns = {name: [] for name in algorithms}
all_lengths = {name: [] for name in algorithms}

for run_idx in range(n_runs):
    seed = 42 + run_idx
    for name, AgentClass in algorithms.items():
        set_seed(seed)
        env = CliffWalkingEnv()
        agent = AgentClass(
            n_states=env.n_states,
            n_actions=env.n_actions,
            gamma=0.99,
            alpha=0.5,
            epsilon=0.1,
            seed=seed,
        )
        returns, lengths = train_agent_on_cliff(agent, env, n_episodes=n_episodes)
        all_returns[name].append(returns)
        all_lengths[name].append(lengths)

# 计算平均和置信区间
mean_returns = {}
std_returns = {}
for name in algorithms:
    arr = np.array(all_returns[name])  # (n_runs, n_episodes)
    mean_returns[name] = np.mean(arr, axis=0)
    std_returns[name] = np.std(arr, axis=0)

# 绘图
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. 学习曲线
ax = axes[0, 0]
window = 20
for name in algorithms:
    mean = mean_returns[name]
    std = std_returns[name]
    smoothed = np.convolve(mean, np.ones(window)/window, mode="valid")
    ci = np.convolve(std, np.ones(window)/window, mode="valid")
    x = range(window-1, len(mean))
    ax.plot(x, smoothed, color=colors[name], linewidth=2, label=name)
    ax.fill_between(x, smoothed - ci/np.sqrt(n_runs), smoothed + ci/np.sqrt(n_runs),
                    alpha=0.15, color=colors[name])
ax.set_xlabel("Episode"); ax.set_ylabel("Return")
ax.set_title("Learning Curves (All Algorithms)")
ax.legend(); ax.grid(True, alpha=0.3)

# 2. 最终性能条形图
ax = axes[0, 1]
final_means = [np.mean(mean_returns[name][-50:]) for name in algorithms]
final_stds = [np.std(mean_returns[name][-50:]) for name in algorithms]
ax.bar(range(len(algorithms)), final_means, yerr=final_stds,
       color=[colors[n] for n in algorithms], capsize=5, alpha=0.8)
ax.set_xticks(range(len(algorithms)))
ax.set_xticklabels(list(algorithms.keys()), rotation=15)
ax.set_ylabel("Average Return (last 50 eps)")
ax.set_title("Final Performance Comparison")
ax.grid(True, alpha=0.3, axis="y")

# 3. 参数敏感性：alpha
ax = axes[1, 0]
alphas = [0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
set_seed(42)
for name, AgentClass in algorithms.items():
    alpha_means = []
    for alpha in alphas:
        env = CliffWalkingEnv()
        agent = AgentClass(
            n_states=env.n_states, n_actions=env.n_actions,
            gamma=0.99, alpha=alpha, epsilon=0.1, seed=42,
        )
        returns, _ = train_agent_on_cliff(agent, env, n_episodes=200)
        alpha_means.append(np.mean(returns[-50:]))
    ax.plot(alphas, alpha_means, 'o-', color=colors[name], linewidth=2, label=name)
ax.set_xlabel("Learning Rate α"); ax.set_ylabel("Avg Return (last 50)")
ax.set_title("Sensitivity to Learning Rate")
ax.legend(); ax.grid(True, alpha=0.3)

# 4. 参数敏感性：epsilon
ax = axes[1, 1]
epsilons = [0.01, 0.05, 0.1, 0.2, 0.5, 0.75]
set_seed(42)
for name, AgentClass in algorithms.items():
    eps_means = []
    for eps in epsilons:
        env = CliffWalkingEnv()
        agent = AgentClass(
            n_states=env.n_states, n_actions=env.n_actions,
            gamma=0.99, alpha=0.5, epsilon=eps, seed=42,
        )
        returns, _ = train_agent_on_cliff(agent, env, n_episodes=200)
        eps_means.append(np.mean(returns[-50:]))
    ax.plot(epsilons, eps_means, 'o-', color=colors[name], linewidth=2, label=name)
ax.set_xlabel("Exploration Rate ε"); ax.set_ylabel("Avg Return (last 50)")
ax.set_title("Sensitivity to Exploration Rate")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/figures/08_all_algorithms_comparison.png")
plt.close()
print("✅ 四种算法综合对比图已保存")


## 11. 总结对比

### 四种算法一览

| 特性 | SARSA | Q-Learning | Expected SARSA | Double Q-Learning |
|------|-------|-----------|----------------|-------------------|
| **类型** | 同策略 (On-policy) | 异策略 (Off-policy) | 同策略 | 异策略 |
| **更新 target** | $R + \gamma Q(S', A')$ | $R + \gamma \max_a Q(S', a)$ | $R + \gamma \mathbb{E}_\pi[Q(S', \cdot)]$ | $R + \gamma Q_B(S', \arg\max_a Q_A(S', a))$ |
| **方差** | 高 | 中 | **低** | 中 |
| **最大化偏差** | 无 | 有 | 部分 | **几乎无** |
| **保守程度** | **最保守** | 冒险 | 介于之间 | 介于之间 |
| **CliffWalking 路径** | 上方安全绕行 | 悬崖边缘 | 略偏离悬崖 | 略偏离悬崖 |
| **收敛速度** | 快（稳定） | 快（可能震荡） | 最快 | 较慢（双表更新） |
| **适用场景** | 安全关键应用 | 需要最优解 | 方差敏感问题 | 高噪环境 |

### 算法关系图谱

```
                    SARSA（同策略，高方差）
                   /       ↓          \
                  /   Expected SARSA    \
                 /    （降低方差）        \
    行为策略 ← — — — — — — — — — — — — → 目标策略
                 \                        /
                  \    Q-Learning（异策略）
                   \       ↓            /
                    Double Q-Learning
                  （消除最大化偏差）
```


<details>
<summary><b>🎯 面试考点 — 点击展开</b></summary>

### 基础概念

**Q1: SARSA 和 Q-Learning 的核心区别是什么？**

> SARSA 是同策略算法，使用实际采取的 $A'$ 来更新；Q-Learning 是异策略算法，使用 $\max_a Q(S', a)$ 来更新，学习的是最优策略而非行为策略。

**Q2: 为什么 CliffWalking 中 SARSA 沿安全路径而 Q-Learning 沿悬崖边缘？**

> SARSA 的更新包含实际动作的后果（含 ε 探索可能掉下悬崖），所以学会了避免危险区域。Q-Learning 假设下一步是最优动作（不掉下悬崖），故敢于靠近悬崖边缘以缩短路径。

**Q3: 什么是同策略和异策略？各自优缺点？**

> 同策略：评估和改进是同一策略（如 SARSA），稳定但样本效率低。异策略：评估的是目标策略，行为策略可不同（如 Q-Learning），可复用历史数据但需考虑分布偏移。

### 进阶理解

**Q4: Expected SARSA 为什么能降低方差？**

> SARSA 的 target 包含 $Q(S', A')$ 这一采样值，$A'$ 的随机性引入方差。Expected SARSA 用 $\mathbb{E}_\pi[Q(S', \cdot)]$ 代替之，相当于对 $A'$ 求平均，消除采样噪声。

**Q5: 最大化偏差是什么？Double Q-Learning 如何解决？**

> $\mathbb{E}[\max_a Q(S', a)] \geq \max_a \mathbb{E}[Q(S', a)]$，导致 Q-Learning 系统性高估。Double Q-Learning 将「选择」和「评估」分开用独立的 Q 表，$a^*$ 由 $Q_A$ 选择，但价值由 $Q_B$ 评估，打破高估循环。

**Q6: 在什么场景下 Expected SARSA 优于 Q-Learning？**

> 当奖励有较高噪声或环境随机性大时，Expected SARSA 的低方差特性使学习更稳定。当需要安全探索时也更好（如 CliffWalking 的安全路径）。

### 实战问题

**Q7: 如果 ε=0，SARSA 和 Q-Learning 会一样吗？**

> 当 ε=0 时，策略退化为贪婪。SARSA 的 $A'$ 就是最优动作 $\arg\max_a Q(S', a)$，所以 $Q(S', A') = \max_a Q(S', a)$。两者更新公式等价。但在实践中 ε=0 意味着无探索，算法可能收敛到次优解。

**Q8: Double Q-Learning 相比 Q-Learning 的代价是什么？**

> 存储和计算的代价翻倍（两个 Q 表），且收敛速度可能稍慢（每个样本只更新一个 Q 表）。但在高噪声环境中这个代价通常值得。

</details>


## 12. 练习

### 基础练习

1. **推导 Expected SARSA 的 target**：对 ε-greedy 策略，写出 Expected SARSA 的更新公式展开式，并证明当 ε→0 时它退化为 Q-Learning。

2. **修改探索率**：在 CliffWalking 中用 ε=0.3 重跑 SARSA 和 Q-Learning，观察策略变化。为什么 SARSA 在高 ε 下表现更差？

3. **实现 n-step SARSA**：在 `TabularSARSAAgent` 的基础上实现 n-step SARSA，公式为：
   $$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ G_t^{(n)} - Q(S_t, A_t) \right]$$
   其中 $G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n}, A_{t+n})$

### 进阶练习

4. **随机 CliffWalking**：给 CliffWalking 添加 slippery 机制（以 20% 概率随机执行非预期动作），比较四种算法的鲁棒性。

5. **Q 值偏差分析**：在训练过程中记录 Q-Learning 和 Double Q-Learning 对最优动作的 Q 值估计，与真实值的对比曲线（可用 MC 估计近似真实值）。

6. **学习率调度**：为各算法实现学习率衰减 $\alpha_t = 1 / t^{0.5}$，观察对收敛速度和最终性能的影响。

### 思考题

7. 如果 CliffWalking 的悬崖奖励从 -100 改为 -10，SARSA 和 Q-Learning 的行为会如何变化？为什么？

8. 在 Double Q-Learning 中，如果 Q_A 和 Q_B 的初始值不同，会有什么影响？

9. Expected SARSA 是否可以和 Double 思想结合？设计 Double Expected SARSA 算法。


## 参考文献

1. **Sutton & Barto (2018)**. Reinforcement Learning: An Introduction (2nd Edition). Chapters 6-7.
2. **Watkins & Dayan (1992)**. Q-Learning. *Machine Learning*, 8(3), 279-292.
3. **Rummery & Niranjan (1994)**. On-line Q-Learning using Connectionist Systems. *Technical Report CUED/F-INFENG/TR 166*.
4. **Van Seijen et al. (2009)**. A Theoretical and Empirical Analysis of Expected Sarsa. *IEEE ADPRL*.
5. **Hasselt (2010)**. Double Q-Learning. *NeurIPS*.
6. **Thrun & Schwartz (1993)**. Issues in Using Function Approximation for Reinforcement Learning. *Proceedings of the Fourth Connectionist Models Summer School*.
